# Model comparison

Historical diagnostics: the legacy sensitivity critic and a paired FLORIS gradient variant. The current generalized-critic EnvPG is in `envpg.ipynb`. All scores use the same WFCRL reward, 25 wind bins and 150 control steps.


In [ ]:
from pathlib import Path
import json
import os
import sys

ROOT = Path.cwd().resolve()
assert (ROOT / "envpg.py").exists(), "Open from the repository root"
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / "__simul__" / "matplotlib"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(ROOT / "wfcrl-benchmark" / "algos"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
from floris.tools.visualization import plot_turbines_with_fi, visualize_cut_plane
from envpg import Actor, REFERENCE, layout, windrose
from wfcrl import environments as envs
from extractors import VectorExtractor
from baseline_ippo import Agent as IPPOAgent
from baseline_mappo import Agent as MAPPOAgent

ENVPG = ("Legacy critic", "Paired FLORIS")
RUNS = {
    ENVPG[0]: ROOT / "runs/envpg-001",
    ENVPG[1]: ROOT / "runs/envpg-mc-paired-100-seed1",
    "IPPO": ROOT / "wfcrl-benchmark/runs/Dec_Ablaincourt_Floris__baseline_ippo__windrose_1__1794149416",
    "MAPPO": ROOT / "wfcrl-benchmark/runs/Dec_Ablaincourt_Floris__baseline_mappo__windrose_1__1794150208",
}
ACTOR_ONLY = {
    seed: ROOT / f"runs/envpg-actor-hybrid-seed{seed}" for seed in (1, 2, 3)
}
FIXED_RUNS = {f"Fixed layout, seed {seed}": run for seed, run in ACTOR_ONLY.items()}
ORDER = ("No yaw", *ENVPG, "No yaw (paired layout)", "IPPO", "MAPPO")
for name, run in {**RUNS, **ACTOR_ONLY}.items():
    assert run.is_dir(), f"Missing {name}: {run}"
config = json.loads((RUNS[ENVPG[0]] / "config.json").read_text())["parameters"]
H = config["horizon"]
WIND_DATA = ROOT / "wfcrl-benchmark/data/smarteole.csv"
for name in ENVPG:
    saved = json.loads((RUNS[name] / "config.json").read_text())["parameters"]
    assert saved["horizon"] == H and saved["load_coef"] == config["load_coef"]
    assert Path(saved["wind_data"]).name == WIND_DATA.name
    assert saved["iterations"] == 100
    assert torch.load(RUNS[name] / "latest.pt", map_location="cpu", weights_only=False)["iteration"] == 99
for name in ("IPPO", "MAPPO"):
    saved = yaml.load((RUNS[name] / "config.yaml").read_text(), Loader=yaml.BaseLoader)
    assert saved["env_id"] == "Dec_Ablaincourt_Floris"
    assert saved["scenario"] == "windrose"
    assert int(saved["episode_length"]) == H + 1
    assert float(saved["load_coef"]) == config["load_coef"]
    assert Path(saved["wind_data"]).name == WIND_DATA.name
print(f"{H} steps/bin, load coefficient {config['load_coef']}")


## Training

The actor curve is a policy-gradient surrogate evaluated on new trajectories each iteration. Evaluation cost is minus the fixed-windrose reward; lower is better.


In [ ]:
events = {}
for name, run in {**{name: RUNS[name] for name in ENVPG}, **ACTOR_ONLY}.items():
    events[name] = EventAccumulator(str(run / "tb"))
    events[name].Reload()

def scalar(name, tag):
    return pd.DataFrame([(e.step, e.value) for e in events[name].Scalars(tag)],
                        columns=("step", "value"))

def evaluation_curve(name):
    reward = scalar(name, "eval/windrose_score").rename(columns={"value": "reward"})
    calls = scalar(name, "simulator/wake_calls").rename(columns={"value": "total_calls"})
    curve = reward.merge(calls, on="step")
    evaluation_calls = 25 * (H + 1)
    if name == ENVPG[0]:
        curve["training_calls"] = curve.total_calls - np.arange(1, len(curve) + 1) * evaluation_calls
    else:
        curve["training_calls"] = (curve.total_calls - evaluation_calls
                                   - np.arange(1, len(curve) + 1) * 2 * evaluation_calls)
    return curve

fig, ax = plt.subplots(1, 3, figsize=(13, 3))
for name in ENVPG:
    curve = evaluation_curve(name)
    ax[0].plot(curve.training_calls, curve.reward, marker="o", markersize=2, label=name)
    ax[1].plot(curve.training_calls, -curve.reward, marker="o", markersize=2, label=name)
ax[0].set(title="Windrose reward", xlabel="Training wake calls", ylabel="Weighted reward")
ax[1].set(title="Evaluation cost", xlabel="Training wake calls", ylabel="− weighted reward")
for name in ("IPPO", "MAPPO"):
    farm = pd.read_csv(RUNS[name] / "farm.csv")
    ax[2].plot(farm["index"], farm.episode_reward / H, label=name)
ax[2].set(title="WFCRL training", xlabel="Environment steps", ylabel="Reward / step")
for a in ax:
    a.legend(fontsize=7)
    a.grid(alpha=.2)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(1, 2, figsize=(9, 3))
for a, tag, title in zip(ax,
                         ("eval/layout_gain_no_yaw", "eval/controller_gain"),
                         ("Layout gain", "Yaw controller gain")):
    curve = scalar(ENVPG[1], tag)
    a.plot(curve.step, curve.value, marker="o", markersize=2)
    a.axhline(0, color="black", linewidth=.7)
    a.set(title=title, xlabel="Design iteration", ylabel="Weighted reward change")
    a.grid(alpha=.2)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(len(ENVPG), 3, figsize=(12, 6))
for row, name in enumerate(ENVPG):
    for a, tag, title in zip(ax[row],
                             ("loss/actor", "loss/critic", "loss/cost_sensitivity"),
                             ("Actor surrogate", "Critic TD loss", "Cost-sensitivity loss")):
        curve = scalar(name, tag)
        a.plot(curve.step, curve.value)
        a.set(title=f"{name}: {title}", xlabel="Design iteration")
        a.grid(alpha=.2)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(1, 2, figsize=(9, 3))
for name in ("IPPO", "MAPPO"):
    losses = pd.read_csv(RUNS[name] / "losses.csv")
    for a, suffix in zip(ax, ("policy_loss", "value_loss")):
        columns = [c for c in losses if c.endswith(suffix)]
        if columns:
            a.plot(losses["index"], losses[columns].mean(axis=1), label=name)
for a, title in zip(ax, ("PPO policy loss", "PPO value loss")):
    a.set(title=title, xlabel="Environment steps")
    a.legend()
    a.grid(alpha=.2)
plt.tight_layout()
plt.show()


## Final scores

The final policies run in the same multi-agent FLORIS environment.


In [ ]:
def load_model(name):
    env = envs.make("Dec_Ablaincourt_Floris", controls=["yaw"], log=False,
                    max_num_steps=H + 1, load_coef=config["load_coef"])
    model = {"env": env, "layout": REFERENCE.copy()}
    if name == "No yaw":
        return model
    if name == "No yaw (paired layout)":
        state = torch.load(RUNS[ENVPG[1]] / "latest.pt", map_location="cpu", weights_only=False)
        model["layout"][1:] += np.asarray(state["layout"]).reshape(6, 2)
    elif name in ENVPG or name in FIXED_RUNS:
        run = RUNS[name] if name in ENVPG else FIXED_RUNS[name]
        state = torch.load(run / "latest.pt", map_location="cpu", weights_only=False)
        saved = json.loads((run / "config.json").read_text())["parameters"]
        actor = Actor(H, saved["max_displacement"],
                      target_tracking=saved.get("target_tracking", False),
                      design_dim=len(state["layout"])).eval()
        actor.load_state_dict(state["actor"])
        model.update(actor=actor, z=np.asarray(state["layout"]))
        if len(model["z"]) == 4:
            model["layout"] = layout(model["z"])
        else:
            model["layout"][1:] += model["z"].reshape(6, 2)
    else:
        saved = yaml.load((RUNS[name] / "config.yaml").read_text(), Loader=yaml.BaseLoader)
        first = env.possible_agents[0]
        observation = VectorExtractor(env.observation_space(first))
        action = VectorExtractor(env.action_space(first))
        cls = IPPOAgent if name == "IPPO" else MAPPOAgent
        agents = []
        for i in range(env.num_turbines):
            agent = cls(observation.space, action.space,
                        tuple(map(int, saved["hidden_layer_nn"]))).eval()
            path = RUNS[name] / f"baseline_{name.lower()}.cleanrl_model_{i}"
            agent.load_state_dict(torch.load(path, map_location="cpu", weights_only=True))
            agents.append(agent)
        model.update(agents=agents, observation=observation, action=action)
    env.mdp.interface.fi.reinitialize(layout_x=model["layout"][:, 0],
                                      layout_y=model["layout"][:, 1])
    return model

models = {name: load_model(name) for name in (*ORDER, *FIXED_RUNS)}


In [ ]:
def command_for(name, model, turbine, index, joint_action):
    if name.startswith("No yaw"):
        return {"yaw": np.zeros(1, dtype=np.float32)}
    if name in ENVPG or name in FIXED_RUNS:
        return {"yaw": np.array([joint_action[index]], dtype=np.float32)}
    obs = torch.tensor(model["observation"](model["env"].observe(turbine)),
                       dtype=torch.float32)
    agent = model["agents"][index]
    if name == "IPPO":
        action = agent.get_action_and_value(obs, deterministic=True)[0]
    else:
        action = agent.get_action(obs, deterministic=True)[0]
    return model["action"].make_dict(action)

def run_case(name, direction, speed):
    model = models[name]
    env = model["env"]
    env.reset(options={"wind_speed": float(speed), "wind_direction": float(direction)})
    first = env.possible_agents[0]
    rows = []
    with torch.no_grad():
        for _ in range(H):
            joint_action = None
            if name in ENVPG or name in FIXED_RUNS:
                usage = [env.accumulated_actions[t]["yaw"] for t in env.possible_agents]
                state = np.r_[env._state["freewind_measurements"], env._state["yaw"],
                              usage, env.num_moves]
                joint_action = model["actor"].sample(
                    torch.tensor(state, dtype=torch.float32)[None],
                    torch.tensor(model["z"], dtype=torch.float32)[None],
                    deterministic=True)[0].numpy()
            for i, turbine in enumerate(env.possible_agents):
                env.step(command_for(name, model, turbine, i, joint_action))
            rows.append({"reward": float(env.rewards[first].item()),
                         "power_mw": sum(env.infos[t]["power"] for t in env.possible_agents),
                         "load_proxy": np.mean(np.abs([env.infos[t]["load"]
                                                       for t in env.possible_agents])),
                         "yaw": np.asarray(env._state["yaw"]).copy()})
    assert env.truncations[first], f"{name} did not complete {H} steps"
    return rows


In [ ]:
conditions = windrose(WIND_DATA)
assert len(conditions) == 25
assert abs(sum(weight for _, _, weight in conditions) - 1) < 1e-9

scores = []
for name in ORDER:
    for direction, speed, weight in conditions:
        rows = run_case(name, direction, speed)
        scores.append({"model": name, "weight": weight,
                       "reward": sum(r["reward"] for r in rows),
                       "power_mw": np.mean([r["power_mw"] for r in rows]),
                       "load_proxy": np.mean([r["load_proxy"] for r in rows])})
frame = pd.DataFrame(scores)
metrics = ("reward", "power_mw", "load_proxy")
summary = pd.DataFrame({key: (frame[key] * frame.weight).groupby(frame.model,
                            sort=False).sum() for key in metrics})
summary["reward_vs_no_yaw"] = summary.reward - summary.loc["No yaw", "reward"]
display(summary.round(3))

fig, ax = plt.subplots(1, 3, figsize=(13, 4))
for a, key, title in zip(ax, metrics, ("Weighted reward / 150 steps",
                                       "Farm power (MW)", "Load proxy")):
    a.barh(summary.index, summary[key])
    a.set_title(title)
    a.grid(axis="x", alpha=.2)
plt.tight_layout()
plt.show()

layout_gain = summary.loc["No yaw (paired layout)", "reward"] - summary.loc["No yaw", "reward"]
yaw_gain = summary.loc[ENVPG[1], "reward"] - summary.loc["No yaw (paired layout)", "reward"]
print(f"Paired FLORIS: layout gain {layout_gain:+.3f}; yaw gain {yaw_gain:+.3f}")

fixed_layout = []
for seed in ACTOR_ONLY:
    name = f"Fixed layout, seed {seed}"
    reward = sum(weight * sum(row["reward"] for row in run_case(name, direction, speed))
                 for direction, speed, weight in conditions)
    fixed_layout.append({"seed": seed, "final_reward": reward,
                         "yaw_gain": reward - summary.loc["No yaw", "reward"]})
display(pd.DataFrame(fixed_layout).set_index("seed").round(3))

for name in ENVPG:
    curve = scalar(name, "eval/windrose_score")
    shift = np.linalg.norm(models[name]["layout"][1:] - REFERENCE[1:], axis=1).max()
    print(f"{name}: evaluation {curve.value.iloc[0]:.1f} → {curve.value.iloc[-1]:.1f}; training wake calls {evaluation_curve(name).training_calls.iloc[-1]:.0f}; largest shift {shift:.2f} m")


## One wind condition

The traces and wakes use 8 m/s from 270° after 150 control steps. The score table uses all 25 bins.


In [ ]:
example = {name: run_case(name, 270.0, 8.0) for name in ORDER}
fig, ax = plt.subplots(1, 3, figsize=(13, 3))
for name, rows in example.items():
    for a, key in zip(ax, ("reward", "power_mw", "load_proxy")):
        a.plot([r[key] for r in rows], label=name)
for a, title in zip(ax, ("Reward / step", "Farm power (MW)", "Load proxy")):
    a.set(title=title, xlabel="Control step")
    a.grid(alpha=.2)
ax[0].legend(fontsize=7)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(1, len(ORDER), figsize=(3 * len(ORDER), 3))
for a, (name, rows) in zip(ax, example.items()):
    yaw = np.stack([r["yaw"] for r in rows]).T
    image = a.imshow(yaw, aspect="auto", origin="lower", cmap="coolwarm",
                     vmin=-40, vmax=40)
    a.set(title=f"{name} yaw (°)", xlabel="Control step", ylabel="Turbine")
    a.set_yticks(range(7), range(1, 8))
fig.colorbar(image, ax=ax, label="Yaw (°)")
plt.show()


In [ ]:
fig, ax = plt.subplots(len(ORDER), 1, figsize=(11, 3 * len(ORDER)), sharex=True, sharey=True)
for a, name in zip(ax, ORDER):
    fi = models[name]["env"].mdp.interface.fi
    yaw = fi.floris.farm.yaw_angles.copy()
    plane = fi.calculate_horizontal_plane(height=90, x_resolution=100, y_resolution=50,
                                          wd=[270.0], ws=[8.0], yaw_angles=yaw)
    image = visualize_cut_plane(plane, ax=a, min_speed=0, max_speed=8,
                                levels=np.linspace(0, 8, 21))
    plot_turbines_with_fi(fi, ax=a, wd=np.array([270.0]), yaw_angles=yaw)
    for i, (x, y) in enumerate(models[name]["layout"]):
        a.text(x, y + 65, str(i + 1), ha="center", fontsize=8)
    a.set(title=name, ylabel="y (m)")
ax[-1].set_xlabel("x (m)")
fig.colorbar(image, ax=ax, label="Wind speed (m/s)")
plt.show()
for model in models.values():
    model["env"].close()
